In [26]:
import tensorflow as tf
import os
import numpy as np
import pandas as pd
from datetime import datetime

"""
Cow Frontal Face Dataset Evaluation
Dataset: 459 cattle identities, 2,847 images
Source: /home/student/cow_frontal_face_dataset/INDIVIDUAL SUBJECTS Data
"""

np.random.seed(42)
tf.random.set_seed(42)

2026-02-08 15:22:25.707601: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-08 15:22:25.936431: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-08 15:22:25.975288: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-08 15:22:26.604199: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; 

In [27]:
# CONFIGURATION

# Paths
TRAINING_DATASET_ROOT = "/home/student/Desktop/Kumar/dataset"
FRONTAL_DATASET_ROOT = "/home/student/cow_frontal_face_dataset/INDIVIDUAL SUBJECTS Data"

MODEL_PATH = "cattle_face_curriculum_saved_model"

class CFG:
    IMG_SIZE = 224


print("\n Dataset Info:")
print(f"  Training data:  {TRAINING_DATASET_ROOT}")
print(f"  Frontal faces:  {FRONTAL_DATASET_ROOT}")
print(f"  Model:          {MODEL_PATH}")


 Dataset Info:
  Training data:  /home/student/Desktop/Kumar/dataset
  Frontal faces:  /home/student/cow_frontal_face_dataset/INDIVIDUAL SUBJECTS Data
  Model:          cattle_face_curriculum_saved_model


In [32]:
# DATA LOADING


def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (CFG.IMG_SIZE, CFG.IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img

class CattleDatasetBuilder:
    """Handles cattle dataset loading"""
    
    def __init__(self, folder, mode='training'):
        self.cattle_data = {}
        self.mode = mode
        
        if mode == 'training':
            self._load_training_data(folder)
        elif mode == 'frontal':
            self._load_frontal_data(folder)
    
    def _load_training_data(self, folder):
        """Load from face_train, face_test, face_val"""
        for subfolder in ['face_train', 'face_test', 'face_val']:
            subfolder_path = os.path.join(folder, subfolder)
            if not os.path.exists(subfolder_path):
                continue
            
            for cow_id in sorted(os.listdir(subfolder_path)):
                cow_path = os.path.join(subfolder_path, cow_id)
                if not os.path.isdir(cow_path):
                    continue
                
                imgs = [os.path.join(cow_path, f) for f in os.listdir(cow_path)
                       if f.lower().endswith((".jpg", ".jpeg", ".png"))]
                
                if len(imgs) > 0:
                    if cow_id in self.cattle_data:
                        self.cattle_data[cow_id].extend(imgs)
                    else:
                        self.cattle_data[cow_id] = imgs
        
        self.classes = sorted(list(self.cattle_data.keys()))
        self.num_classes = len(self.classes)
        self.total_images = sum(len(v) for v in self.cattle_data.values())
    
    def _load_frontal_data(self, folder):
        """
        Load from frontal face dataset
        Structure: /INDIVIDUAL SUBJECTS Data/cattle-XXX/images
        """
        print(f"\n Loading frontal face dataset from: {folder}")
        
        if not os.path.exists(folder):
            raise ValueError(f"Dataset root not found: {folder}")
        
        # List all cattle folders
        cattle_folders = sorted([
            item for item in os.listdir(folder)
            if os.path.isdir(os.path.join(folder, item))
            and item.startswith('cattle-')
        ])
        
        print(f"  Found {len(cattle_folders)} cattle folders")
        
        # Load images from each cattle folder
        for cattle_id in cattle_folders:
            cattle_path = os.path.join(folder, cattle_id)
            
            # Get all images from this cattle folder
            # Images might be directly in folder or in subfolders
            cattle_images = []
            
            # Strategy 1: Check for images directly in cattle folder
            direct_images = [
                os.path.join(cattle_path, f)
                for f in os.listdir(cattle_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ]
            cattle_images.extend(direct_images)
            
            # Strategy 2: Check for subfolders with images
            subfolders = [
                item for item in os.listdir(cattle_path)
                if os.path.isdir(os.path.join(cattle_path, item))
            ]
            
            for subfolder in subfolders:
                subfolder_path = os.path.join(cattle_path, subfolder)
                subfolder_images = [
                    os.path.join(subfolder_path, f)
                    for f in os.listdir(subfolder_path)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
                ]
                cattle_images.extend(subfolder_images)
            
            if len(cattle_images) > 0:
                self.cattle_data[cattle_id] = cattle_images
        
        self.classes = sorted(list(self.cattle_data.keys()))
        self.num_classes = len(self.classes)
        self.total_images = sum(len(v) for v in self.cattle_data.values())
        
        print(f"\n Frontal face dataset loaded:")
        print(f"  Total cattle: {self.num_classes}")
        print(f"  Total images: {self.total_images}")
        print(f"  Avg images/cattle: {self.total_images / self.num_classes:.1f}")
        
        # Show distribution
        image_counts = [len(imgs) for imgs in self.cattle_data.values()]
        print(f"  Min images: {min(image_counts)}")
        print(f"  Max images: {max(image_counts)}")
        print(f"  Median images: {np.median(image_counts):.0f}")

In [33]:
# LOAD TRAINING DATA (FOR GALLERY)

print("STEP 1: LOADING TRAINING DATA (GALLERY)")

train_data = CattleDatasetBuilder(TRAINING_DATASET_ROOT, mode='training')
print(f"\n Training gallery: {train_data.num_classes} cattle, {train_data.total_images} images")

STEP 1: LOADING TRAINING DATA (GALLERY)

 Training gallery: 468 cattle, 4726 images


In [34]:
# LOAD FRONTAL FACE DATA

print("STEP 2: LOADING EXTERNAL DATASET")

frontal_data = CattleDatasetBuilder(FRONTAL_DATASET_ROOT, mode='frontal')

# Check for overlap
overlap = set(train_data.cattle_data.keys()) & set(frontal_data.cattle_data.keys())
if len(overlap) > 0:
    print(f"\n  WARNING: Found {len(overlap)} overlapping IDs")
    print(f"   Sample overlaps: {list(overlap)[:5]}")
    print("   These are likely different cattle with similar naming")
else:
    print(f"\n No ID overlap - truly external cattle")

STEP 2: LOADING EXTERNAL DATASET

 Loading frontal face dataset from: /home/student/cow_frontal_face_dataset/INDIVIDUAL SUBJECTS Data
  Found 459 cattle folders

 Frontal face dataset loaded:
  Total cattle: 459
  Total images: 2849
  Avg images/cattle: 6.2
  Min images: 1
  Max images: 38
  Median images: 5

 No ID overlap - truly external cattle


In [35]:
# LOAD MODEL

print("STEP 3: LOADING TRAINED MODEL")

try:
    loaded_model = tf.saved_model.load(MODEL_PATH)
    infer_fn = loaded_model.signatures["serving_default"]
    input_name = list(infer_fn.structured_input_signature[1].keys())[0]
    output_name = list(infer_fn.structured_outputs.keys())[0]
    print(" Model loaded successfully")
except Exception as e:
    print(f" Error loading model: {e}")
    raise

def get_embedding(img_path):
    """Extract embedding from single image"""
    try:
        img = load_image(img_path)[None]
        output = infer_fn(**{input_name: img})
        emb = output[output_name].numpy()[0]
        # Normalize
        emb = emb / (np.linalg.norm(emb) + 1e-8)
        return emb
    except Exception as e:
        # Silent fail for individual images
        return None

STEP 3: LOADING TRAINED MODEL


2026-02-08 15:24:54.834225: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-08 15:24:55.481045: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13103 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:d8:00.0, compute capability: 7.5


 Model loaded successfully


In [36]:
# BUILD GALLERY

print("STEP 4: BUILDING TRAINING GALLERY")

gallery = {}
gallery_build_stats = []

for cow_id, img_list in train_data.cattle_data.items():
    embeddings = []
    for img_path in img_list:
        emb = get_embedding(img_path)
        if emb is not None:
            embeddings.append(emb)
    
    if len(embeddings) > 0:
        avg_emb = np.mean(embeddings, axis=0)
        avg_emb = avg_emb / (np.linalg.norm(avg_emb) + 1e-8)
        gallery[cow_id] = avg_emb
        gallery_build_stats.append({
            'Cattle_ID': cow_id,
            'Images_Used': len(embeddings)
        })

print(f" Gallery built: {len(gallery)} training cattle enrolled")
print(f"  Total images used: {sum([s['Images_Used'] for s in gallery_build_stats])}")

STEP 4: BUILDING TRAINING GALLERY


2026-02-08 15:25:12.485956: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8201


 Gallery built: 468 training cattle enrolled
  Total images used: 4726


In [37]:
# EXTRACT FRONTAL EMBEDDINGS

print("STEP 5: EXTRACTING EXTERNAL DATASET'S FRONTAL FACE EMBEDDINGS")

print(f"Processing {frontal_data.num_classes} cattle...")

frontal_records = []
failed_images = []
processed_cattle = 0

for cow_id, img_list in frontal_data.cattle_data.items():
    processed_cattle += 1
    
    # Progress updates for large dataset
    if processed_cattle % 50 == 0:
        print(f"  Progress: {processed_cattle}/{frontal_data.num_classes} cattle ({processed_cattle/frontal_data.num_classes*100:.1f}%)")
    
    for img_path in img_list:
        emb = get_embedding(img_path)
        if emb is not None:
            frontal_records.append((cow_id, img_path, emb))
        else:
            failed_images.append(img_path)

print(f"\n Extracted {len(frontal_records)} embeddings from {frontal_data.num_classes} frontal cattle")
if len(failed_images) > 0:
    print(f"  Failed to process {len(failed_images)} images ({len(failed_images)/frontal_data.total_images*100:.1f}%)")

STEP 5: EXTRACTING EXTERNAL DATASET'S FRONTAL FACE EMBEDDINGS
Processing 459 cattle...
  Progress: 50/459 cattle (10.9%)
  Progress: 100/459 cattle (21.8%)
  Progress: 150/459 cattle (32.7%)
  Progress: 200/459 cattle (43.6%)
  Progress: 250/459 cattle (54.5%)
  Progress: 300/459 cattle (65.4%)
  Progress: 350/459 cattle (76.3%)
  Progress: 400/459 cattle (87.1%)
  Progress: 450/459 cattle (98.0%)

 Extracted 2849 embeddings from 459 frontal cattle


In [38]:
# EVALUATION

print("EVALUATION: Frontal Face Dataset vs Training Gallery")

print(f"\n  IMPORTANT: All {frontal_data.num_classes} cattle are UNKNOWN")
print(f"  Expected: High rejection rate (low FAR)")
print(f"  This is a LARGE-SCALE open-set test")

# Test at multiple thresholds
thresholds = np.arange(0.5, 0.95, 0.03)
threshold_results = []

print("\nComputing FAR across thresholds...")

for tau in thresholds:
    stats = {
        "total": 0,
        "false_accept": 0,
        "reject": 0,
        "scores": []
    }
    
    for cow_id, img_path, emb in frontal_records:
        scores = {gid: np.dot(emb, gallery[gid]) for gid in gallery}
        best_match = max(scores, key=scores.get)
        best_score = scores[best_match]
        
        stats["total"] += 1
        stats["scores"].append(best_score)
        
        if best_score >= tau:
            stats["false_accept"] += 1
        else:
            stats["reject"] += 1
    
    far = (stats["false_accept"] / stats["total"]) * 100 if stats["total"] > 0 else 0
    reject_rate = (stats["reject"] / stats["total"]) * 100 if stats["total"] > 0 else 0
    
    threshold_results.append({
        "threshold": tau,
        "false_accepts": stats["false_accept"],
        "rejects": stats["reject"],
        "total": stats["total"],
        "FAR_%": far,
        "reject_rate_%": reject_rate,
        "avg_score": np.mean(stats["scores"]) if stats["scores"] else 0,
        "max_score": np.max(stats["scores"]) if stats["scores"] else 0
    })

results_df = pd.DataFrame(threshold_results)

print("\nFalse Accept Rate (FAR) Across Thresholds:")
print("=" * 70)
print("Threshold | FAR    | False Accepts | Rejects | Avg Score | Max Score")
print("=" * 70)
for _, row in results_df.iterrows():
    print(f"{row['threshold']:.2f}      | {row['FAR_%']:5.1f}% | "
          f"{row['false_accepts']:4.0f}          | {row['rejects']:4.0f}    | "
          f"{row['avg_score']:.4f}    | {row['max_score']:.4f}")

# Find optimal threshold
optimal_idx = results_df['FAR_%'].idxmin()
optimal = results_df.iloc[optimal_idx]

print("\n" + "=" * 70)
print(f"Best threshold (lowest FAR): {optimal['threshold']:.2f}")
print(f"  FAR: {optimal['FAR_%']:.1f}%")
print(f"  False accepts: {optimal['false_accepts']:.0f}/{optimal['total']:.0f}")
print(f"  Correct rejects: {optimal['rejects']:.0f}/{optimal['total']:.0f}")
print(f"  Avg score: {optimal['avg_score']:.4f}")
print("=" * 70)

EVALUATION: Frontal Face Dataset vs Training Gallery

  IMPORTANT: All 459 cattle are UNKNOWN
  Expected: High rejection rate (low FAR)
  This is a LARGE-SCALE open-set test

Computing FAR across thresholds...

False Accept Rate (FAR) Across Thresholds:
Threshold | FAR    | False Accepts | Rejects | Avg Score | Max Score
0.50      |  25.1% |  716          | 2133    | 0.4479    | 0.7088
0.53      |  15.0% |  428          | 2421    | 0.4479    | 0.7088
0.56      |   8.2% |  235          | 2614    | 0.4479    | 0.7088
0.59      |   3.8% |  108          | 2741    | 0.4479    | 0.7088
0.62      |   1.4% |   41          | 2808    | 0.4479    | 0.7088
0.65      |   0.6% |   17          | 2832    | 0.4479    | 0.7088
0.68      |   0.1% |    4          | 2845    | 0.4479    | 0.7088
0.71      |   0.0% |    0          | 2849    | 0.4479    | 0.7088
0.74      |   0.0% |    0          | 2849    | 0.4479    | 0.7088
0.77      |   0.0% |    0          | 2849    | 0.4479    | 0.7088
0.80      |   0.0

In [39]:
# DETAILED ANALYSIS


print("\n" + "=" * 80)
print(f"DETAILED ANALYSIS @ Threshold {optimal['threshold']:.2f}")
print("=" * 80)

print("\nGenerating detailed results...")

detailed_results = []

for cow_id, img_path, emb in frontal_records:
    scores = {gid: np.dot(emb, gallery[gid]) for gid in gallery}
    best_match = max(scores, key=scores.get)
    best_score = scores[best_match]
    
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top3_matches = sorted_scores[:3]
    
    decision = "FALSE_ACCEPT" if best_score >= optimal['threshold'] else "CORRECT_REJECT"
    
    detailed_results.append({
        "Frontal_Cattle": cow_id,
        "Top1_Match": best_match,
        "Top1_Score": best_score,
        "Top2_Match": top3_matches[1][0] if len(top3_matches) > 1 else "",
        "Top2_Score": top3_matches[1][1] if len(top3_matches) > 1 else 0,
        "Top3_Match": top3_matches[2][0] if len(top3_matches) > 2 else "",
        "Top3_Score": top3_matches[2][1] if len(top3_matches) > 2 else 0,
        "Decision": decision,
        "Image": os.path.basename(img_path)
    })

detailed_df = pd.DataFrame(detailed_results)

# Per-cattle summary
print("Computing per-cattle statistics...")

per_cattle = detailed_df.groupby("Frontal_Cattle").agg(
    Total=("Decision", "count"),
    False_Accepts=("Decision", lambda x: (x == "FALSE_ACCEPT").sum()),
    Correct_Rejects=("Decision", lambda x: (x == "CORRECT_REJECT").sum()),
    FAR_pct=("Decision", lambda x: (x == "FALSE_ACCEPT").sum() / len(x) * 100 if len(x) > 0 else 0),
    Avg_Top1_Score=("Top1_Score", "mean"),
    Max_Top1_Score=("Top1_Score", "max"),
    Min_Top1_Score=("Top1_Score", "min"),
    Most_Confused_With=("Top1_Match", lambda x: x.mode()[0] if len(x.mode()) > 0 else "")
)

print("\nPer Frontal Cattle Summary (first 20):")
print("=" * 120)
print(per_cattle.head(20).to_string())

print("\nCattle with highest FAR (top 10):")
top_far = per_cattle.nlargest(10, 'FAR_pct')
print(top_far[['Total', 'False_Accepts', 'FAR_pct', 'Avg_Top1_Score']].to_string())

print("\nCattle with 100% correct rejection (bottom 10 by FAR):")
perfect_reject = per_cattle[per_cattle['FAR_pct'] == 0.0].head(10)
if len(perfect_reject) > 0:
    print(perfect_reject[['Total', 'Avg_Top1_Score', 'Max_Top1_Score']].to_string())
else:
    print("  (None - all cattle had at least one false accept)")

# Score distribution
print("\n\n Score Distribution Statistics:")
print("=" * 70)
all_scores = detailed_df["Top1_Score"].values
print(f"   Mean:   {np.mean(all_scores):.4f}")
print(f"   Median: {np.median(all_scores):.4f}")
print(f"   Std:    {np.std(all_scores):.4f}")
print(f"   Min:    {np.min(all_scores):.4f}")
print(f"   Max:    {np.max(all_scores):.4f}")
print(f"   Q1:     {np.percentile(all_scores, 25):.4f}")
print(f"   Q3:     {np.percentile(all_scores, 75):.4f}")

# False accepts
fa_df = detailed_df[detailed_df["Decision"] == "FALSE_ACCEPT"]
print(f"\n\nFalse Accepts Detail ({len(fa_df)} cases @ τ={optimal['threshold']:.2f}):")
print(f"  {len(fa_df)} / {len(detailed_df)} images ({len(fa_df)/len(detailed_df)*100:.1f}%)")

if len(fa_df) > 0:
    print("\nFirst 20 false accepts:")
    print(fa_df[["Frontal_Cattle", "Top1_Match", "Top1_Score", "Top2_Match", "Top2_Score"]].head(20).to_string(index=False))
    
    # Analyze which training cattle get most false matches
    print("\nTraining cattle receiving most false matches:")
    fa_match_counts = fa_df['Top1_Match'].value_counts().head(10)
    print(fa_match_counts.to_string())
else:
    print("   ✓ NO FALSE ACCEPTS - PERFECT REJECTION!")


# SAVE RESULTS


print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
excel_name = f"frontal_face_eval_{timestamp}.xlsx"

print("Writing Excel file (may take a moment for large dataset)...")

with pd.ExcelWriter(excel_name, engine="openpyxl") as writer:
    # Summary
    summary_data = {
        "Metric": [
            "Model",
            "Training Cattle (Gallery)",
            "Training Images",
            "Frontal Face Dataset Cattle",
            "Frontal Face Dataset Images",
            "External Source",
            "",
            "Best Threshold",
            "FAR at Best Threshold",
            "False Accepts",
            "Correctly Rejected",
            "",
            "Avg External Score",
            "Max External Score",
            "Min External Score",
            "Score Std Dev",
            "",
            "Cattle with 0% FAR",
            "Cattle with 100% FAR",
            "Cattle with >50% FAR"
        ],
        "Value": [
            "Cattle Face Recognition (Curriculum Learning)",
            str(train_data.num_classes),
            str(train_data.total_images),
            str(frontal_data.num_classes),
            str(len(frontal_records)),
            "Cow Frontal Face Dataset (459 cattle)",
            "",
            f"{optimal['threshold']:.2f}",
            f"{optimal['FAR_%']:.1f}%",
            f"{optimal['false_accepts']:.0f}/{optimal['total']:.0f}",
            f"{optimal['rejects']:.0f}/{optimal['total']:.0f}",
            "",
            f"{np.mean(all_scores):.4f}",
            f"{np.max(all_scores):.4f}",
            f"{np.min(all_scores):.4f}",
            f"{np.std(all_scores):.4f}",
            "",
            str(len(per_cattle[per_cattle['FAR_pct'] == 0.0])),
            str(len(per_cattle[per_cattle['FAR_pct'] == 100.0])),
            str(len(per_cattle[per_cattle['FAR_pct'] > 50.0]))
        ]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name="Summary", index=False)
    
    # Threshold sweep
    results_df.to_excel(writer, sheet_name="Threshold_Sweep", index=False)
    
    # Detailed results (may be large)
    detailed_df.to_excel(writer, sheet_name="Frontal_Details", index=False)
    
    # Per-cattle summary
    per_cattle.to_excel(writer, sheet_name="Per_Frontal_Cattle")
    
    # False accepts
    if len(fa_df) > 0:
        fa_df.to_excel(writer, sheet_name="False_Accepts", index=False)
    
    # Gallery info
    pd.DataFrame(gallery_build_stats).to_excel(writer, sheet_name="Gallery_Info", index=False)

print(f"✓ Saved to: {excel_name}")


# FINAL INTERPRETATION



print("FINAL INTERPRETATION")

print(f"\n Evaluation Scenario:")
print(f"   Model trained on: {train_data.num_classes} cattle (custom dataset)")
print(f"   Tested on:        {frontal_data.num_classes} cattle (frontal face dataset)")
print(f"   → LARGE-SCALE open-set evaluation!")
print(f"   → {frontal_data.num_classes} unknown identities")

print(f"\n Performance:")
print(f"   Best FAR: {optimal['FAR_%']:.1f}% @ threshold {optimal['threshold']:.2f}")
print(f"   False accepts: {optimal['false_accepts']:.0f} images")
print(f"   Correct rejects: {optimal['rejects']:.0f} images")
print(f"   Average similarity: {np.mean(all_scores):.4f}")

if optimal['FAR_%'] == 0.0:
    print(f"\n PERFECT: Model rejects all {frontal_data.num_classes} unknown cattle!")
    print(f"   → Exceptional open-set performance")
elif optimal['FAR_%'] < 5.0:
    print(f"\n EXCELLENT: FAR = {optimal['FAR_%']:.1f}%")
    print(f"   → Very strong open-set performance on large dataset")
elif optimal['FAR_%'] < 15.0:
    print(f"\n VERY GOOD: FAR = {optimal['FAR_%']:.1f}%")
    print(f"   → Strong open-set performance")
elif optimal['FAR_%'] < 30.0:
    print(f"\n GOOD: FAR = {optimal['FAR_%']:.1f}%")
    print(f"   → Acceptable open-set performance for {frontal_data.num_classes} unknowns")
else:
    print(f"\n  FAIR: FAR = {optimal['FAR_%']:.1f}%")
    print(f"   → Challenging with {frontal_data.num_classes} unknown cattle")

print(f"\n Key Insights:")
num_perfect = len(per_cattle[per_cattle['FAR_pct'] == 0.0])
num_problematic = len(per_cattle[per_cattle['FAR_pct'] > 50.0])

print(f"   Cattle with 0% FAR: {num_perfect}/{frontal_data.num_classes} ({num_perfect/frontal_data.num_classes*100:.1f}%)")
print(f"   Cattle with >50% FAR: {num_problematic}/{frontal_data.num_classes} ({num_problematic/frontal_data.num_classes*100:.1f}%)")

print(f"\n   Average similarity: {np.mean(all_scores):.4f}")
if np.mean(all_scores) < 0.4:
    print(f"   → Excellent separation from training cattle")
elif np.mean(all_scores) < 0.5:
    print(f"   → Good separation from training cattle")
elif np.mean(all_scores) < 0.6:
    print(f"   → Moderate separation from training cattle")
else:
    print(f"   → Weak separation - frontal cattle similar to training set")

print("EVALUATION COMPLETE")
print(f"\n Results saved to: {excel_name}")


DETAILED ANALYSIS @ Threshold 0.71

Generating detailed results...
Computing per-cattle statistics...

Per Frontal Cattle Summary (first 20):
                Total  False_Accepts  Correct_Rejects  FAR_pct  Avg_Top1_Score  Max_Top1_Score  Min_Top1_Score Most_Confused_With
Frontal_Cattle                                                                                                                   
cattle-001         10              0               10      0.0        0.446221        0.488659        0.387773         HF-2-C1016
cattle-002          5              0                5      0.0        0.455583        0.528393        0.404215     HFCROS-6-C1017
cattle-003          3              0                3      0.0        0.393989        0.441572        0.348877         HF-6-C8016
cattle-004          5              0                5      0.0        0.454319        0.519043        0.363099         HF-7-C5088
cattle-005          3              0                3      0.0        0.45693